# Full run, step by step

This notebook runs the stress engine on one input set, exactly as `uv run stress run` does, and then shows what each step R1–R11 produced. Every table is read back from the sealed run folder `runs/<run_id>/`, so what you see is what the run sealed. Nothing is edited. The few checks recompute an identity from the run's own tables.

| Step | What it does | Tables shown |
|---|---|---|
| R1 | Open the run | config, versions |
| R2 | Load inputs | the seven datasets in `data/` |
| R3 | Validate and route | `rejections`, starting-point reconciliation |
| R4 | Scenarios and Z-score | `indices`, `driver_contributions`, `drivers` |
| R5 | Contractual cash flows | `cf_schedule` |
| R6 | Parameters | `pit_matrices`, `pd_curves`, `lgd_of_year`, `parameters` |
| R7 | Project the year | `slices`, `stage_flows`, `grade_flows` |
| R8 | Year-end provisions | `provisions`, `ecl_buckets` |
| R9 | Stock to flow | `charges` |
| R10 | Reports | `reports/*.csv`, `summary.md` |
| R11 | Seal the run | `manifest.json` |

One facility is followed from its cash flows (R5) to its charge (R9). Use the project kernel (`.venv`); setup is in [README.md](README.md).

## Settings

A new run takes about 30 seconds and writes a new folder under `runs/`. To look at a run that already exists, set `RUN_ID` to its folder name or to `"latest"`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm

import nbtools as nb

INPUTS = "inputs/d62d0386f2c5256e"  # the current input set; the table below lists them all
SCENARIOS = ("baseline", "adverse")
VARIANT = "core"  # core, annuity, linear or utilisation
RUN_ID = None  # None starts a new run; "latest" or a run ID opens a sealed one
FACILITY = None  # the facility followed from R5 to R9; None picks one with an undrawn line

nb.setup()
nb.input_sets()

## Run R1–R11

The output is the engine's step log: a line when each step starts and when it ends, with its duration. R7–R9 repeat for each scenario and projected year. R8 also values the opening book once, as a diagnostic next to the frozen opening allowances. The full log, with times, is `run.log` in the run folder.

In [ ]:
if RUN_ID is None:
    run = nb.run_engine(INPUTS, SCENARIOS, VARIANT)
else:
    run = nb.open_run(RUN_ID, variant=VARIANT)

ps = run.data("parameter_set")
grades = run.grades()
start_year = int(ps["horizon"]["snapshot_year"])
years = [int(y) for y in ps["horizon"]["projection_years"]]
first = years[0]
scenarios = run.scenarios
stressed = scenarios[-1]  # the scenario shown where only one fits
names = run.portfolio_names()
run

## R1 · Open the run

R1 fixes the input set by its content hash and records the code, spec and parameter-set versions. Every preparation run is labelled SYNTHETIC, and regulatory mode blocks in v1. The run ID ends in a hash of the config, the input set, the code version and the spec version.

In [ ]:
display(run.overview().to_frame())
print(run.config_yaml())

## R2 · Load inputs

R2 reads the seven datasets, checks every file against its hash in the input-set manifest, and records row counts. The run keeps a copy in `data/`, which is what is read here. No later step reads a file.

In [ ]:
loaded = pd.Series(run.manifest["load_controls"], name="rows loaded by R2")
in_data = pd.Series({name: len(run.data(name)) for name in loaded.index if name != "scenario_values"},
                    name="rows in data/")
pd.concat([loaded, in_data], axis=1)

The book at the starting point, by portfolio. Amounts in EUR millions; rates, maturities and LGDs are simple means.

In [ ]:
book = run.book()
undrawn = run.data("commitments").groupby("facility_id")["nominal_undrawn"].sum()
book["undrawn"] = book["facility_id"].map(undrawn).fillna(0.0)

by_stage = book.pivot_table(index="portfolio", columns="ifrs_stage", values="gca", aggfunc="sum",
                            fill_value=0.0).rename(columns=lambda s: f"S{s} GCA")
portfolios = book.groupby("portfolio").agg(
    facilities=("facility_id", "size"), gca=("gca", "sum"), undrawn=("undrawn", "sum"),
    mean_rate=("interest_rate", "mean"), mean_maturity=("residual_maturity", "mean"),
    mean_ttc_lgd=("ttc_lgd", "mean"),
).join(by_stage)
portfolios.insert(0, "name", portfolios.index.map(names))
print(f"{len(book):,} facilities; GCA EUR {book['gca'].sum() / nb.MILLION:,.1f}m; "
      f"undrawn EUR {book['undrawn'].sum() / nb.MILLION:,.1f}m")
means = {"mean_rate": "{:.2%}", "mean_maturity": "{:.1f}", "mean_ttc_lgd": "{:.1%}"}
nb.millions(portfolios, ["gca", "undrawn", *by_stage.columns]).format(means, subset=list(means))

The parameter set per segment, and the master scale. These values are proposed, not owner-approved (assumption A2): ρ is the asset correlation in the Z shift, `sicr_k` the SICR multiple, `lgd_w` the LGD sensitivity to Z, and `cpr_ttc` and `cpr_v` the prepayment rate and its sensitivity.

In [ ]:
print(f"{ps['parameter_set_id']} version {ps['version']}, status {ps['status']}")
segments = pd.DataFrame({s: {k: v for k, v in p.items() if k not in ("z_drivers", "portfolio")}
                         for s, p in ps["segments"].items()}).T
display(segments)
pd.Series(ps["master_scale"]["ttc_pd"], index=grades, name="TTC one-year PD").to_frame().T.style.format("{:.2%}")

### The facility followed through R5–R9

By default, the first facility with an undrawn commitment and three to six years to maturity. Its segment and country set which matrices and curves are shown below.

In [ ]:
if FACILITY is None:
    committed = book[book["undrawn"] > 0].sort_values("facility_id")
    in_range = committed[committed["residual_maturity"].between(3, 6)]
    FACILITY = (in_range if len(in_range) else committed).iloc[0]["facility_id"]
facility = book.set_index("facility_id").loc[FACILITY]
segment, country = facility["segment"], facility["country"]
facility[["portfolio", "segment", "country", "product", "ifrs_stage", "current_grade",
          "origination_grade", "gca", "undrawn", "interest_rate", "rate_type", "residual_maturity",
          "repayment_profile", "ttc_lgd"]].to_frame(FACILITY)

## R3 · Validate and route

R3 checks the schema, referential integrity, the EUR-only and AC-only scope, every TTC matrix, and whether each facility's grade explains its reported stage. A rejected record goes to `rejections.parquet` with a reason code and never reaches a total.

In [ ]:
rejections = run.table("rejections")
print(f"{len(rejections):,} rejected records; manifest: {run.manifest['rejections']}")
if len(rejections):
    display(rejections.groupby(["record_type", "reason_code"]).size().rename("records").to_frame())

The starting point reconciled, per portfolio and stage. `engine_2026_<scenario>_on` and `_off` are the engine's own end-2026 valuation, a diagnostic next to the frozen O4 allowance (assumption A16). Charges always start from the frozen allowance. EUR millions.

In [ ]:
start = run.report("starting_point_reconciliation")
amounts = ["loaded_gca", "rejected_gca", "accepted_gca", "accepted_undrawn", "converted_commitments",
           "opening_allowance_on", "opening_provision_off",
           *[c for c in start.columns if c.startswith("engine_2026_")]]
view = start.set_index(["portfolio", "stage"])[["loaded_count", "accepted_count", *amounts]]
nb.millions(view, amounts)

## R4 · Scenarios and Z-score

R4 compounds the property price changes into indices. It then standardises each macro driver and combines the results into one Z per segment, country, year and scenario: Z = Σ β · (x − μ) / σ. A positive Z means a better economy than average. The same Z drives the PD matrix, LGD and prepayment. After the last scenario year, adverse reverts linearly to the baseline Z, and both then stay flat until the last bucket year.

In [ ]:
indices = run.table("indices")
print(f"Property price indices, {country}")
indices[indices["country"] == country].pivot_table(index="year", columns=["variable", "scenario"],
                                                   values="index")

In [ ]:
contrib = run.table("driver_contributions")
z = run.table("drivers")
mine = contrib[(contrib["segment"] == segment) & (contrib["country"] == country)]
print(f"Driver contributions to Z, {segment} {country}")
display(mine.drop(columns=["segment", "country"]).set_index(["scenario", "year", "driver"]))

key = ["segment", "country", "scenario", "year"]
total = contrib.groupby(key)["contribution"].sum()
gap = (z.set_index(key)["z"].reindex(total.index) - total).abs().max()
spread = z.groupby(["segment", "scenario", "year"])["z"].nunique().max()
print(f"largest |Z - sum of contributions|: {gap:.1e}")
print(f"distinct Z values across countries for one segment, scenario and year: {spread} (A5)")

In [ ]:
last_moving = int(z.loc[z["z_source"] != "flat", "year"].max())
path = z[(z["segment"] == segment) & (z["country"] == country) & (z["year"] <= last_moving + 1)]
print(f"Z of {segment} {country}: scenario path, reversion, then flat until {z['year'].max()}")
path.pivot(index="year", columns="scenario", values=["z", "z_source", "adverse_weight"])

In [ ]:
shown = z[(z["country"] == country) & (z["year"] <= last_moving + 2)]
segment_ids = sorted(z["segment"].unique())
fig, axes = plt.subplots(2, (len(segment_ids) + 2) // 2, figsize=(11, 4.6), sharex=True, sharey=True)
for ax, seg in zip(axes.flat, segment_ids):
    for s in scenarios:
        line = shown[(shown["segment"] == seg) & (shown["scenario"] == s)]
        ax.plot(line["year"], line["z"], color=nb.SCENARIO_COLOURS[s], label=s)
    ax.axhline(0, color=nb.AXIS, linewidth=0.8)
    ax.axvspan(first - 0.5, years[-1] + 0.5, color=nb.GRID, alpha=0.5, linewidth=0)
    ax.set_title(seg, loc="left")
for ax in list(axes.flat)[len(segment_ids):]:
    ax.axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", bbox_to_anchor=(0.97, 0.18))
fig.suptitle(f"Z by segment ({country}; shaded: scenario years, then reversion)", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## R5 · Contractual cash flows

R5 is the only step that creates cash flows. From balance, rate and maturity it builds a yearly contractual schedule to maturity: opening balance, interest, principal, closing balance and the discount factor from the same rate. The schedule is the same in every scenario. At each valuation year-end it restarts from the static stock balance (A15), so each facility has one schedule per valuation year. Under `core` every facility is bullet (A11).

In [ ]:
schedule = run.table("cf_schedule")
print(f"{len(schedule):,} schedule rows for {schedule['facility_id'].nunique():,} facilities")
display(schedule.groupby(["valuation_year", "profile"]).agg(
    facilities=("facility_id", "nunique"), periods=("period", "size")))
ends = schedule.sort_values("period").groupby(["facility_id", "valuation_year"]).tail(1)
print("largest closing balance at maturity (every schedule ends at zero): "
      f"{ends['closing_balance'].abs().max():.2e}")

In [ ]:
print(f"Schedule of {FACILITY} from end-{start_year}")
schedule[(schedule["facility_id"] == FACILITY) & (schedule["valuation_year"] == start_year)]

## R6 · Parameters

R6 builds every scenario-dependent value for each facility, scenario and default bucket.

- **PD:** each segment's TTC matrix is shifted by the year's Z into a point-in-time (PiT) matrix M(y). The cumulative PD F[k] is the default column of M(y+1) × … × M(y+k).
- **LGD:** Φ(Φ⁻¹(LGD_TTC) − w · Z).
- **EAD:** the contractual balance from R5 times the share not yet prepaid, plus undrawn × CCF.

### PiT matrices

In [ ]:
ttc = run.data("ttc_pd_matrix")
pit = run.table("pit_matrices")


def matrix(df):
    m = df.pivot_table(index="from_grade", columns="to_grade", values="probability")
    return m.reindex(index=nb.grade_order(m.index, grades), columns=nb.grade_order(m.columns, grades))


ttc_m = matrix(ttc[ttc["segment"] == segment])
pit_m = {s: matrix(pit[(pit["segment"] == segment) & (pit["country"] == country)
                       & (pit["scenario"] == s) & (pit["year"] == first)]) for s in scenarios}
row_sums = pit.groupby(["segment", "country", "scenario", "year", "from_grade"])["probability"].sum()
print(f"{len(row_sums):,} PiT rows; largest |row sum - 1|: {(row_sums - 1).abs().max():.1e}")
print(f"One-year PD by grade, {segment} {country}: TTC and PiT {first}")
pd.DataFrame({"TTC": ttc_m["D"], **{f"PiT {s}": m["D"] for s, m in pit_m.items()}}).drop(
    index="D").style.format("{:.3%}")

How the stressed year moves probability mass, as PiT minus TTC in percentage points. Red cells gain probability and blue cells lose it, so under stress the diagonal loses mass to downgrades and to D.

In [ ]:
shift = pit_m[stressed] - ttc_m
limit = shift.abs().max().max()
print(f"PiT {stressed} {first} minus TTC, {segment} {country}")
shift.style.format("{:+.2%}").background_gradient(cmap=nb.DIVERGING, vmin=-limit, vmax=limit, axis=None)

### PD curves

Cumulative PD from the starting point for a low grade, the followed facility's grade and a high grade. In the PD-03 identities, the marginal PD is m[k] = S[k−1] · q[k], survival is S[k] = S[k−1] − m[k], and the cumulative PD is F[k] = 1 − S[k]. Here they are checked on every bucket. The curves hold whole years only: a final bucket shorter than a year takes d × its whole-year marginal PD in the ECL step (A7).

In [ ]:
curves = run.table("pd_curves")
full = curves.sort_values(
    ["segment", "country", "scenario", "valuation_year", "cohort", "default_bucket"])
prior = full.groupby(["segment", "country", "scenario", "valuation_year", "cohort"])["s"].shift(1).fillna(1.0)
print(f"PD-03 on {len(full):,} buckets: "
      f"max |m - (S[k-1] - S)| = {(full['m'] - (prior - full['s'])).abs().max():.1e}, "
      f"max |F - (1 - S)| = {(full['f'] - (1 - full['s'])).abs().max():.1e}, "
      f"max |m - S[k-1] q| = {(full['m'] - prior * full['q']).abs().max():.1e}")

picked = nb.grade_order({grades[1], facility["current_grade"], grades[-2]}, grades)
mine = full[(full["segment"] == segment) & (full["country"] == country)
            & (full["valuation_year"] == start_year)]
fig, axes = plt.subplots(1, len(picked), figsize=(11, 3.2))
for ax, grade in zip(axes, picked):
    for s in scenarios:
        c = mine[(mine["cohort"] == grade) & (mine["scenario"] == s)]
        ax.plot(c["bucket_year"], c["f"], color=nb.SCENARIO_COLOURS[s], label=s)
    ax.axvspan(first - 0.5, years[-1] + 0.5, color=nb.GRID, alpha=0.5, linewidth=0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f"{grade}" + (" (followed facility)" if grade == facility["current_grade"] else ""),
                 loc="left")
axes[0].legend(loc="upper left")
fig.suptitle(f"Cumulative PD from end-{start_year}, {segment} {country}", x=0.01, ha="left")
fig.tight_layout()
plt.show()

### LGD

The exposure-weighted LGD per segment and year, then the formula checked on the followed facility.

In [ ]:
lgd = run.report("lgd_by_z")
display(lgd.pivot_table(index="segment", columns=["scenario", "year"],
                        values="exposure_weighted_lgd").style.format("{:.2%}"))

w = ps["segments"][segment]["lgd_w"]
mine = run.table("lgd_of_year")
mine = mine[mine["facility_id"] == FACILITY].assign(
    formula=lambda d: norm.cdf(norm.ppf(facility["ttc_lgd"]) - w * d["z"]))
print(f"{FACILITY}: TTC LGD {facility['ttc_lgd']:.2%}, w = {w}; "
      f"largest |engine - formula|: {(mine['lgd'] - mine['formula']).abs().max():.1e}")
mine.set_index(["scenario", "year"])

### EAD and the ParameterRecords

The facility's records for the stressed scenario from the starting point: one row per default bucket. Prepayment shortens the loss profile through the not-prepaid share and never changes the stock. The checks run on every record.

In [ ]:
params = run.table("parameters")
undrawn_ead = params["undrawn"] * params["ccf"].fillna(0.0)
print(f"{len(params):,} records; largest gaps: "
      f"drawn vs balance x not-prepaid {(params['ead_drawn'] - params['balance'] * params['not_prepaid_share']).abs().max():.1e}, "
      f"undrawn vs undrawn x CCF {(params['ead_undrawn'] - undrawn_ead).abs().max():.1e}, "
      f"total vs drawn + undrawn {(params['ead_total'] - params['ead_drawn'] - params['ead_undrawn']).abs().max():.1e}")
records = params[(params["facility_id"] == FACILITY) & (params["scenario"] == stressed)
                 & (params["valuation_year"] == start_year)]
records[["default_bucket", "bucket_year", "fraction", "z", "lgd", "prepayment_rate",
         "not_prepaid_share", "balance", "discount_factor", "undrawn", "ccf", "ead_drawn",
         "ead_undrawn", "ead_total"]]

## R7 · Project the year

Each year R7 moves every facility's grade distribution one step: closing = opening × M(y). Whatever reaches D becomes a new S3 cohort, which keeps its entry year and origin stage. S3 never cures. Each grade slice takes its stage from the SICR rule against the facility's origination grade; a facility with a qualitative trigger (30+ days past due, watchlist, forbearance) stays S2 in every grade until it defaults (A30). Total GCA and undrawn stay at end-2026 levels (static balance sheet), and a facility that matures is replaced like for like.

In [ ]:
slices = run.table("slices")
mix = slices.pivot_table(index=["scenario", "year"], columns="stage", values="gca",
                         aggfunc="sum").rename(columns=lambda s: f"S{s}")
mix["total"] = mix.sum(axis=1)
print("GCA by stage, EUR millions")
nb.millions(mix)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, stage in zip(axes, ("S1", "S2", "S3")):
    for s in scenarios:
        line = mix.loc[s, stage] / nb.MILLION
        ax.plot(line.index, line.values, color=nb.SCENARIO_COLOURS[s], label=s, marker="o",
                markersize=5, markeredgecolor=nb.SURFACE, markeredgewidth=1.5)
    ax.set_title(f"{stage} GCA, EUR m", loc="left")
    ax.set_xticks(sorted(mix.index.get_level_values("year").unique()))
axes[0].legend(loc="best")
fig.tight_layout()
plt.show()

Stage flows in the first projected year, in EUR millions. Each flow is a sum of matrix cells applied to the slices, and the S3 row only flows to S3.

In [ ]:
flows = run.table("stage_flows")
nb.millions(flows[flows["year"] == first].pivot_table(
    index=["scenario", "from_stage"], columns="to_stage", values="gca", aggfunc="sum", fill_value=0.0))

In [ ]:
grade_flows = run.table("grade_flows")
g = grade_flows[(grade_flows["segment"] == segment) & (grade_flows["scenario"] == stressed)
                & (grade_flows["year"] == first)].pivot_table(
    index="from_grade", columns="to_grade", values="gca", aggfunc="sum")
g = g.reindex(index=nb.grade_order(g.index, grades), columns=nb.grade_order(g.columns, grades))
g = g[g.sum(axis=1) > 0]  # only the grades the book holds at the start of the year
print(f"Grade flows of {segment}, {stressed} {first}, all countries, EUR millions")
nb.millions(g, digits=2)

The S3 stock by cohort (EUR millions): the original S3 book, plus one cohort per entry year and origin stage. Then the static books, which must not move.

In [ ]:
cohorts = run.report("s3_cohorts")
display(nb.millions(cohorts.pivot_table(index=["scenario", "cohort"], columns="year", values="gca",
                                        aggfunc="sum")))
books = run.report("static_books").groupby(["scenario", "year"])[
    ["gca", "undrawn", "converted", "gca_plus_converted"]].sum()
nb.millions(books)

## R8 · Year-end provisions

- **S1:** the 12-month ECL, Σ m[k] · EAD[k] · LGD[k] · DF[k] over the next bucket. At end-2029 it uses the interpolated 2030 loss rate.
- **S2:** the same sum over the remaining life.
- **Original S3:** max(GCA × LGD of the year, prior allowance), per exposure.
- **New S3 cohort:** booked once at flow × LGD of the default year, then floored.

Drawn amounts go to the on-balance allowance and undrawn amounts to the off-balance provision. Provisions by stage and method, on plus off, EUR millions:

In [ ]:
prov = run.table("provisions")
prov = prov.assign(total=prov["allowance_on"] + prov["provision_off"])
nb.millions(prov.pivot_table(index=["stage", "method"], columns=["scenario", "year"], values="total",
                             aggfunc="sum"))

In [ ]:
cover = run.report("impairment_by_stage").groupby(["scenario", "year", "stage"])[
    ["coverage_numerator", "coverage_denominator"]].sum()
cover = (cover["coverage_numerator"] / cover["coverage_denominator"]).unstack("stage")
print("Coverage (provisions / GCA) by stage")
display(cover.rename(columns=lambda s: f"S{s}").style.format("{:.2%}"))

s3 = prov[prov["stage"] == 3].groupby(["scenario", "year"]).agg(
    raw_on=("raw_on", "sum"), allowance_on=("allowance_on", "sum"),
    slices=("slice", "size"), floor_binding=("floor_binding", "sum"))
print("S3: raw against floored allowance (EUR millions), and how often the floor binds")
nb.millions(s3, ["raw_on", "allowance_on"])

In [ ]:
buckets = run.table("ecl_buckets")
b = buckets[(buckets["segment"] == segment) & (buckets["stage"] == 2)
            & (buckets["valuation_year"] == first)].pivot_table(
    index="default_bucket", columns="scenario", values="ecl_on")
print(f"S2 lifetime ECL of {segment} at end-{first} by default bucket (first ten), EUR millions; "
      "R10 checks that the buckets sum to the provisions")
nb.millions(b.head(10), digits=2)

## R9 · Stock to flow

Charge(y) = closing − opening, separately for the on-balance allowance and the off-balance provision. The first opening is the frozen O4 allowance. FX movement is zero, and overlays are carried unchanged. EUR millions:

In [ ]:
charges = run.table("charges")
cols = ["opening_on", "closing_on", "charge_on", "opening_off", "closing_off", "charge_off"]
bridge = charges.groupby(["scenario", "year"])[cols].sum()
bridge["charge_total"] = bridge["charge_on"] + bridge["charge_off"]
gap = max((charges["closing_on"] - charges["opening_on"] - charges["charge_on"]).abs().max(),
          (charges["closing_off"] - charges["opening_off"] - charges["charge_off"]).abs().max())
frozen = run.data("opening_provisions")[["allowance_on", "provision_off"]].sum().sum()
opening = charges[charges["year"] == first].groupby("scenario")[["opening_on", "opening_off"]].sum().sum(axis=1)
print(f"largest |closing - opening - charge| on one facility: {gap:.1e}")
print(f"first opening per scenario minus frozen O4 allowance: {(opening - frozen).abs().max():.1e}")
nb.millions(bridge)

In [ ]:
charge = bridge["charge_total"].unstack("scenario") / nb.MILLION
closing = (bridge["closing_on"] + bridge["closing_off"]).unstack("scenario") / nb.MILLION
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
width = 0.2
for i, s in enumerate(scenarios):
    offset = (i - (len(scenarios) - 1) / 2) * width
    axes[0].bar(charge.index + offset, charge[s], width=width * 0.9, color=nb.SCENARIO_COLOURS[s], label=s)
    axes[1].bar(closing.index + offset, closing[s], width=width * 0.9, color=nb.SCENARIO_COLOURS[s], label=s)
axes[0].axhline(0, color=nb.AXIS, linewidth=0.8)
axes[0].set_title("Impairment charge, EUR m", loc="left")
axes[1].set_title("Closing provisions, EUR m", loc="left")
for ax in axes:
    ax.set_xticks(years)
axes[0].legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
impairment = run.report("impairment")
cumulative = impairment[impairment["year"] == years[-1]].pivot_table(
    index="portfolio", columns="scenario", values="cumulative_charge")
opening_gca = book.groupby("portfolio")["gca"].sum()
opening_gca["Total"] = opening_gca.sum()
share = cumulative.div(opening_gca, axis=0)
print(f"Cumulative charge {first}-{years[-1]} by portfolio: EUR millions, then % of opening GCA")
display(nb.millions(cumulative))
share.style.format("{:.2%}")

## Follow one facility through the years

The followed facility in the stressed scenario: its GCA per slice, the allowance per slice, and its charges. The facility starts in one grade. Each year the PiT matrix spreads it over grades, and the share that reaches D becomes a new S3 cohort named by its entry year and origin stage. Amounts in EUR.

In [ ]:
mine = slices[(slices["facility_id"] == FACILITY) & (slices["scenario"] == stressed)]
order = nb.slice_order(mine["slice"], grades)
stage_of = mine.drop_duplicates("slice").set_index("slice")["stage"].reindex(order)
for value in ("gca", "allowance_on", "provision_off"):
    view = mine.pivot_table(index="slice", columns="year", values=value, aggfunc="sum").reindex(order)
    if not view.abs().to_numpy().any():
        continue
    view.insert(0, "stage", stage_of)
    print(f"{FACILITY} {stressed}: {value} by slice")
    display(view.style.format("{:,.0f}", subset=[c for c in view.columns if c != "stage"], na_rep=""))

In [ ]:
charges[charges["facility_id"] == FACILITY].set_index(["scenario", "year"])[cols]

## R10 · Reports

R10 sums money from the facility and bucket results, keeps the numerator and denominator of every ratio, checks the reconciliations against the §12.3 tolerances, and rounds only here. These are the files it wrote, and the summary:

In [ ]:
reports_dir = run.path / "results" / "reports"
display(pd.DataFrame([{"report": p.name, "kB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(reports_dir.iterdir())]))
Markdown(run.summary())

## R11 · Seal the run

R11 writes `manifest.json` with the hash of every input and output and makes the folder read-only. A rerun from the manifest (`uv run stress rerun --run runs/<run_id>`) must give identical output hashes. The table after the outputs compares this run with every other sealed run of the same input set, scenarios and variant: runs on the same code version must be identical. Only a committed code version names the code exactly; a `+dirty` version means the engine had uncommitted changes when it ran.

In [ ]:
print(f"{len(run.manifest['outputs'])} outputs hashed; digest {run.manifest['outputs_digest']}")
display(run.outputs())
nb.reproduction(run)